---
# COSC2673 | COSC2793 (Computational) Machine Learning
## Week 5 Lab: **Training a Classification Model & Typical ML Pipeline**
---

# Introduction

In previous weeks, we covered data loading, exploratory data analysis (EDA), and data preparation for training machine learning models. This lab focuses on the standard machine learning pipeline using a classification example.

This lab assumes that you have completed prior labs. If you have not, please do so before attempting this lab.

## Objective

- Review Python and machine learning libraries
- Train a classification model
- Practice a typical machine learning workflow

## Dataset
This lab uses the `Cardiotocography Data Set` from UCI [Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Cardiotocography). The dataset contains fetal heart rate (FHR) and uterine contraction (UC) features from cardiotocograms evaluated by obstetricians. 2126 CTGs were processed, feature values were computed, and the data was labeled by consensus fetal state: 0=normal, 1=suspect, 2=pathologic.

The dataset in this lab is a subset with the following columns:

1. LB - FHR baseline (beats per minute)  
2. AC - # of accelerations per second  
3. FM - # of fetal movements per second  
4. UC - # of uterine contractions per second  
5. DL - # of light decelerations per second  
6. DS - # of severe decelerations per second  
7. DP - # of prolongued decelerations per second  
8. ASTV - percentage of time with abnormal short term variability
9. MSTV - mean value of short term variability
10. ALTV - percentage of time with abnormal long term variability
11. MLTV - mean value of long term variability
12. Width - width of FHR histogram
13. Min - minimum of FHR histogram
14. Max - Maximum of FHR histogram
15. Nmax - # of histogram peaks
16. Nzeros - # of histogram zeros
17. Mode - histogram mode
18. Mean - histogram mean
19. Median - histogram median
20. Variance - histogram variance
21. Tendency - histogram tendency
22. TARGET: NSP - fetal state class code (0=normal; 1=suspect; 2=pathologic)

**The goal is to predict whether a fetal sample is 0=normal, 1=suspect, or 2=pathologic.**

Before running the notebook, confirm that `Cardiotocography_Data_Set_subset.csv` exists in the workspace.

# Load the dataset and some cleaning

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

data = pd.read_csv('./Cardiotocography_Data_Set_subset.csv', delimiter=',')
data.head()

Transform the target to match the task

In [ ]:
print(np.unique(data['NSP'].astype(int)))

In [ ]:
data['NSP'] = data['NSP'].astype(int) - 1
print(np.unique(data['NSP']))

Next, check for missing values and decide how to handle them. `describe()` can reveal columns with fewer entries than expected.

In [ ]:
data.describe()

### Observations

- The `Mode` column has 2113 entries while other columns have 2126.

Missing values are typically represented as NaN. Let's check for NaN counts.

In [ ]:
pd.isna(data).sum()

The `Mode` column has 13 NaN values. We can find which instances/rows this corresponds to:

In [ ]:
data[pd.isna(data).any(axis=1)]

### Missing value handling options

- Remove rows with missing values (loses samples).
- Impute with zero or column mean (check if it makes sense for the feature).
- Predict missing values using other features.

In this dataset, `Mode` is strongly correlated with `Median` and `Mean`, so a reasonable approach is to fill missing `Mode` values using `Median`.

In [ ]:
data.loc[pd.isna(data['Mode']), 'Mode'] = data.loc[pd.isna(data['Mode']), 'Median']

In [ ]:
pd.isna(data).sum()

# EDA

This section presents examples of useful exploratory techniques for this dataset, but it is not exhaustive.

**Task:** You learned EDA in Lab 02. Explore additional methods for this problem and do not limit yourself to the examples here.

Begin with pairwise scatter plots to check for patterns between feature pairs.

In [ ]:
import seaborn as sns

g = sns.PairGrid(data, vars=['LB', 'FM' , 'ASTV' , 'ALTV', 'Width', 
                             'Nmax', 'Mode', 'Mean', 'Median', 'Variance'], hue="NSP")
g.map(sns.scatterplot)
plt.show()

What observations did you make?

**Observations:** 
- Some plots show that a non-linear decision boundary might be able to separate the two classes. e.g. ASTV vs ASTL
- Some plots show that a linear decision boundary might be able to separate the two classes. e.g. Median vs Variance

Also examine the correlation heatmap.

In [ ]:
f, ax = plt.subplots(figsize=(11, 9))
corr = data.corr()
ax = sns.heatmap(
    corr, 
    vmin=-1, vmax=1, center=0,
    cmap=sns.diverging_palette(20, 220, n=200),
    square=True
)
ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=90,
    horizontalalignment='right'
);

What observations did you make?

Discuss with your tutor in class.

### Class distribution

Plot the target distribution to inspect class balance.

In [ ]:
data['NSP'].hist(figsize=(5,5))
plt.xlabel('NSP')
plt.ylabel('frequency')
plt.show()

What observations did you make?

Discuss with your tutor in class.

# Typical model development process

A standard machine learning workflow usually includes these steps:

1. **Define objectives**: select the performance metric and target.
2. **Design experiment**: prepare training, validation, and test data plus diagnostics for overfitting/underfitting and feature analysis.
3. **Establish baseline**: choose an initial model, cost function, and optimization strategy.
4. **Iterate**: improve the model with incremental changes such as hyperparameter tuning or algorithm updates based on performance insights.


## Performance metric

Choose a metric appropriate for this multi-class problem, for example `accuracy_score` or `f1_score`. See the scikit-learn metrics documentation for a full list.

EDA insights are important when deciding on a metric. Use class balance and application needs to guide your choice. 

In this problem, we use macro-averaged `f1_score` to give equal weight to all three classes. Target performance is 0.85 macro F1.

**Task:** Read about F1 score: [f1_score](https://sebastianraschka.com/faq/docs/computing-the-f1-score.html).


## Setup the experiment - data splits

Next: determine how to evaluate performance with held-out data.

Common methods:
1. Hold-out validation
2. Cross-validation

Select the method suited to your dataset. For learning, we will demonstrate both approaches.

## Hold-out validation

Split data into three sets:
1. Training: learn model parameters.
2. Validation: tune hyperparameters and select models.
3. Test: evaluate final generalization performance (not used for tuning).

In this example, use 60/20/20 split.

In [ ]:
from sklearn.model_selection import train_test_split

train_data_, test_data = train_test_split(
    data, test_size=0.2, shuffle=True, random_state=0
)
train_data, val_data = train_test_split(
    train_data_, test_size=0.25, shuffle=True, random_state=0
)

print(train_data.shape[0], val_data.shape[0], test_data.shape[0])

Convert the data to NumPy arrays.

In [ ]:
train_X = train_data.drop(['NSP',], axis=1).to_numpy()
train_y = train_data['NSP'].to_numpy()

val_X = val_data.drop(['NSP',], axis=1).to_numpy()
val_y = val_data['NSP'].to_numpy()

test_X = test_data.drop(['NSP',], axis=1).to_numpy()
test_y = test_data['NSP'].to_numpy()

Set up helper functions to evaluate performance.

In [ ]:
from sklearn.metrics import f1_score

def get_f1_scores(clf, train_X, train_y, val_X, val_y):
    train_pred = clf.predict(train_X)
    val_pred = clf.predict(val_X)
    
    train_f1 = f1_score(train_y, train_pred, average='macro')
    val_f1 = f1_score(val_y, val_pred, average='macro')
    
    return train_f1, val_f1

## Baseline model

Choose a baseline model for this task. Here we use regularized polynomial logistic regression.

There may be stronger models available, but logistic regression is a good starting point and offers clear baseline behavior. Use EDA findings to select modeling choices.

Polynomial features allow non-linear boundaries; regularization helps counter multicollinearity and overfitting. 

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(3)
poly.fit(train_X)
train_X = poly.transform(train_X)
test_X = poly.transform(test_X)
val_X = poly.transform(val_X)

Polynomial features should be scaled. Apply MinMax scaling; choose the scaler based on EDA if needed.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(train_X)

train_X = scaler.transform(train_X)
val_X = scaler.transform(val_X)
test_X = scaler.transform(test_X)

Evaluate the unregularized logistic regression to verify the pipeline works. If you get a max_iter warning, increasing max_iter usually fixes it; for now we proceed with the result.

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=0, penalty=None, solver='saga', 
                         max_iter=1000, 
                         class_weight='balanced').fit(train_X, train_y.ravel())

train_f1, val_f1 = get_f1_scores(clf, train_X, train_y, val_X, val_y)
print("Train F1-Score score: {:.3f}".format(train_f1))
print("Validation F1-Score score: {:.3f}".format(val_f1))

Baseline training performance may be good, but you may observe a gap between training and validation scores (generalization gap).

**How to address this gap?**
- Apply regularization, adjust model complexity, or explore more robust models. Start from a baseline and improve iteratively.

## Apply regularisation

Choose a regularization strength (lambda) by model selection methods such as:
1. Grid search
2. Random search

This example uses grid search. Define a set of lambda values, then train and evaluate the model for each value to find the best balance of bias and variance.

In [ ]:
lambda_params = np.logspace(-5, 1, num=25)    # establish the lambda values to test (grid)

# Then search
train_performace = list()
valid_performace = list()

for lambda_param in lambda_params:
    clf = LogisticRegression(penalty='l2', C = 1.0/lambda_param, 
                             random_state=0, max_iter=1000 , 
                             class_weight='balanced').fit(train_X, train_y.ravel())
    
    train_f1, val_f1 = get_f1_scores(clf, train_X, train_y, val_X, val_y)
    
    train_performace.append(train_f1)
    valid_performace.append(val_f1)

In [ ]:
lambda_params

Plot training and validation F1 scores for each lambda value. Use this plot to choose the best lambda and adjust the search range if necessary.

In [ ]:
plt.plot([1.0/lambda_param for lambda_param in lambda_params], 
         [tp for tp in train_performace], 'r-')
plt.plot([1.0/lambda_param for lambda_param in lambda_params], 
         [vp for vp in valid_performace], 'b--')
plt.xscale("log")
plt.ylabel('F1 Score')
plt.xlabel('Model Capacity')
plt.legend(['Training','Validation'])
plt.show()

### Choose lambda

Pick the lambda value with the highest validation performance and a small generalization gap.

In [ ]:
clf = LogisticRegression(penalty='l2', C = 40.0, random_state=0, 
                         max_iter=2000, 
                         class_weight='balanced').fit(train_X, train_y.ravel())

train_f1, val_f1 = get_f1_scores(clf, train_X, train_y, val_X, val_y)
print("Train F1-Score score: {:.3f}".format(train_f1))
print("Validation F1-Score score: {:.3f}".format(val_f1))

### Compare with unregularized model

Assess whether regularization improves validation performance and reduces overfitting.

### Target evaluation

- The target is 0.85 F1. If current performance is below target, consider further feature engineering, model selection, or tuning.

Task: Identify other hyper-parameters and tune them under the hold-out validation framework.  

Discuss your answer with your tutor.

## Testing the model

Once the model is tuned, evaluate it on the held-out test set to estimate generalization performance. Use classification reports and confusion matrices to inspect errors.

In [ ]:
from sklearn.metrics import classification_report

test_pred = clf.predict(test_X)
    
print(classification_report(test_y, test_pred,))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay.from_estimator(clf, test_X, test_y,
                                             cmap=plt.cm.Blues)
plt.show()

What is your conclusion? Are you confident about the model?

We should also use model visualisation techniques discussed in last weeks lab to get a better understanding of the model. Since the techniques were introduced last week, it will be an exercise for you.

Task: Use appropriate visualisation techniques to understand the model you have developed.

## K-Fold cross-validation

Use K-fold cross-validation to estimate model performance across different data splits. Keep a separate test set for final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(data, test_size=0.2, 
                                         shuffle=True, random_state=0)
    
print(train_data.shape[0], test_data.shape[0])

The next couple of steps are the same as we did in hold out validation. 

In [ ]:
train_X = train_data.drop(['NSP',], axis=1).to_numpy()
train_y = train_data[['NSP']].to_numpy()

test_X = test_data.drop(['NSP',], axis=1).to_numpy()
test_y = test_data[['NSP']].to_numpy()

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(3)
poly.fit(train_X)
train_X = poly.transform(train_X)
test_X = poly.transform(test_X)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(train_X)

train_X = scaler.transform(train_X)
test_X = scaler.transform(test_X)

Now apply cross-validation. This example uses 5-fold CV and a reduced max_iter (100) to save runtime; increase max_iter as needed.  

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, make_scorer

f1_scorer = make_scorer(f1_score, average='weighted')
lambda_params = np.logspace(-10, 2, num=5)

cv_results = dict()

for lambda_param in lambda_params:
    clf = LogisticRegression(penalty='l2', C = 1.0/lambda_param, 
                             max_iter=100, 
                             class_weight='balanced')
    
    scores = cross_validate(clf, train_X, train_y.ravel(), 
                            scoring=f1_scorer, return_estimator=True,
                            return_train_score=True, cv=5)
    
    cv_results[lambda_param] = scores

In [ ]:
fig, ax = plt.subplots()

val_means = [np.mean(cv_results[lambda_param]['test_score']) 
             for lambda_param in lambda_params]

val_std = [np.std(cv_results[lambda_param]['test_score']) 
           for lambda_param in lambda_params]

train_means = [np.mean(cv_results[lambda_param]['train_score']) 
               for lambda_param in lambda_params]

train_std = [np.std(cv_results[lambda_param]['train_score']) 
             for lambda_param in lambda_params]

ax.errorbar([1.0/lambda_param for lambda_param in lambda_params], 
            val_means,
            yerr=val_std)

ax.errorbar([1.0/lambda_param for lambda_param in lambda_params], 
            train_means,
            yerr=train_std)

plt.xscale("log")
plt.ylabel('Classification Error')
plt.xlabel('Model Capacity')
plt.legend(['Validation','Training',])
plt.show()

What observations did you make?

What will be the best lambda value?

Task: Identify other hyper-parameters and device a mechanism to tune then under hold-out validation framework.  

Discuss your answer with your tutor.

Task: Test the final model and use appropriate visualisation techniques to understand the model you have developed.

# Exercise: Full Cardiotocography Data Set

Task: Now you see how to do this task with the smaller dataset. Repeat the same process for the complete dataset. Use the full Cardiotocography Dataset in `Cardiotocography_Data_Set.csv` and see if you can develop a better model. 

Task: There are more convenient functions in sklearn to do grid search for hyper-parameter tuning. These are specially useful when there are many hyper-parameters to tune. GridSearchCV function is one example. Read the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV) and understand how it works.